# Part d

In [1]:
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation

pcd_gt = o3d.io.read_point_cloud("ground_truth.ply")
pcd_noisy = o3d.io.read_point_cloud("transformed_associated.ply")

r = np.asarray(pcd_gt.points)
r_hat = np.asarray(pcd_noisy.points)

In [2]:
N = len(r)

# center
r_c = np.mean(r, axis=0)
r_hat_c = np.mean(r_hat, axis=0)

# center the point clouds
r_bar = r - r_c
r_hat_bar = r_hat - r_hat_c

# Run part (b)/(c)
H = r_hat_bar.T @ r_bar
U, S, Vt = np.linalg.svd(H)
R = Vt.T @ U.T

# From part (a)
tau = r_c - R @ r_hat_c

In [3]:
print(r_hat.shape)

(1877661, 3)


In [4]:
# Align other cloud (r_hat)
# Dumb numpy stuff to get this to work vector-wise
r_transformed = (R @ r_hat.T).T

# part (a)
e = r_transformed - r + tau
loss = np.mean(np.sum(e**2, axis=1))

In [5]:
print(f"Optimal translation : {tau}")
print(f"Optimal rotation:\n{R}")
print(f"Loss J(R, tau): {loss}")

Optimal translation : [0.51769478 0.57728611 0.50401143]
Optimal rotation:
[[-0.30702737 -0.9514257   0.02287657]
 [-0.89566072  0.28073815 -0.34493183]
 [ 0.32175468 -0.12639316 -0.93834892]]
Loss J(R, tau): 0.007494166321482214


## Part e

In [6]:
gt_pose = np.loadtxt("gt_pose.csv", delimiter=",", skiprows=1)

qx, qy, qz, qw, px, py, pz = gt_pose
rot_truth = Rotation.from_quat([qx, qy, qz, qw], scalar_first=False)
position = np.array([px, py, pz])

In [7]:
rot_calc = Rotation.from_matrix(R)
R_err = rot_truth * rot_calc.inv()

In [8]:
translation_error = np.linalg.norm(tau - position)
rotation_error_deg = np.degrees(R_err.magnitude())

In [9]:
print(f"Rotation error: {rotation_error_deg}º")
print(f"Translation error: {translation_error}m")

Rotation error: 6.168882851498211e-15º
Translation error: 1.6011864169946884e-15m


In [14]:
print(R)
print(np.linalg.det(R))



[[-0.30702737 -0.9514257   0.02287657]
 [-0.89566072  0.28073815 -0.34493183]
 [ 0.32175468 -0.12639316 -0.93834892]]
1.0000000000000004
